# Wildfire spread forecast: hourly U-Net on GOES-18

CMPE 295 Master's Project, FightFireWithAI (Earl Padron, Isla Shi, Nikhil Koganti, Scott Kennedy)

This notebook produces the map layers for the app:

- current fire: cells where GOES-18 detected fire in the last hour
- spread risk at +1, +3 and +6 hours: one GeoJSON per horizon

Data sources:

- fire detections: GOES-17/18 fire product (every 10 min, about 2 km)
- weather: RTMA hourly analysis for training, Open-Meteo for forecasts
- terrain and vegetation: SRTM elevation/slope and VIIRS NDVI (Earth Engine)
- training fires: large California fires from MTBS, 2019-2024

First run: build the samples, train, then forecast (switches are in the setup cell).
After that only the forecast needs to run. Outputs go to `outputs_v3/` on Drive.

## 1. Setup

In [ ]:
import os, sys, json, math, re, glob
import datetime as dt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_ROOT     = "/content/drive/MyDrive/Colab Notebooks/CMPE 295 Master's Project/next-day-wildfire-spread"
BUNDLE_DIR    = os.path.join(DATA_ROOT, 'unet_bundle_v3')       # trained models live in goes_h1, goes_h3, goes_h6
OUTPUT_DIR    = os.path.join(DATA_ROOT, 'outputs_v3')           # geojson, netcdf, test tables
GOES_DATA_DIR = os.path.join(DATA_ROOT, 'goes_hourly_samples')  # training samples (.npz)
GOES_CACHE    = '/content/goes_cache' if IN_COLAB else '/tmp/goes_cache'
EE_PROJECT    = 'wildfire-prediction-spread'
for d in (BUNDLE_DIR, OUTPUT_DIR, GOES_CACHE):
    os.makedirs(d, exist_ok=True)

# what to run
RUN_GOES_CURRENT_FIRE = True    # current fire layer, no model needed
RUN_GOES_BUILD        = False   # build training samples in Earth Engine (slow, only once)
RUN_GOES_TRAIN        = False   # train the +1/+3/+6 h models
RUN_GOES_FORECAST     = True    # forecast and export geojson

# forecast time and place
GOES_TIME   = None              # None = Palisades replay, 'now' = latest full hour, or e.g. '2025-01-10T21:00Z'
GOES_CENTER = None              # (lat, lon) of the fire, None = Palisades
EVENT = {'name': 'Palisades', 'time': '2025-01-10T21:00Z',
         'pinn_bounds': (120_000.0, 146_000.0, -445_000.0, -429_000.0)}   # PINN domain, EPSG:3310 metres

# grid: 1 km cells on a fixed California-wide lattice, so grid_id always means the same place
GRID_CRS = 'EPSG:3310'
RES_M = 1000.0
LATTICE_X0, LATTICE_Y0, LATTICE_NCOLS = -430_000.0, 480_000.0, 1000
CA_BBOX = (-124.5, 32.5, -114.0, 42.1)
GOES_TILE = 96                  # km, tile size around a fire (multiple of 8)

# GOES-18 on AWS
GOES_BUCKET, GOES_PRODUCT = 'noaa-goes18', 'ABI-L2-FDCC'
GOES_FILE_STRIDE = 2            # every 2nd 5-min scan = 10-min cadence, same as the training data

# training data
GOES_HORIZONS_H   = (1, 3, 6)
GOES_STEP_H       = 3           # one sample every 3 hours of a fire
GOES_EVENT_DAYS   = 14          # days after ignition to sample
GOES_YEARS        = (2019, 2024)
GOES_MIN_ACRES    = 20_000
GOES_MAX_PER_YEAR = 6

# model and training
FILTERS, POS_WEIGHT, LR = 32, 10.0, 1e-3
EPOCHS, PATIENCE, BATCH_SIZE, TRAIN_CROP = 30, 5, 16, 32
MAX_FIRE_DIST_KM = 16

# map output
P_EMIT = 0.05                   # cells below this probability are left out of the geojson
P_MAX = 0.60                    # probabilities are capped here (higher values were over-confident)
MODERATE_TARGET_RECALL = 0.65
WEATHER_GRID_N = 9              # forecast weather: 9x9 Open-Meteo points, interpolated

# PINN ensemble (optional). TODO: confirm units, sign and normalization with the PINN team
W_UNET = 0.5
PINN_CFG = {'checkpoint': os.path.join(DATA_ROOT, 'levelset_pinn_real.pt'),
            't_origin_utc': '2025-01-07T21:04:00Z', 'T_days': 4.53,
            'x0': None, 'y0': None, 'L_m': None,     # x_n = (x - x0) / L, phi in metres = L * phi_n
            'hidden': 128, 'layers': 4, 'inside_negative': True, 's_m': 1000.0}

In [ ]:
# Earth Engine login (training samples and terrain/NDVI for forecasts)
import ee
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

In [ ]:
import subprocess, importlib
for pkg in ('pyproj', 'xarray'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
if importlib.util.find_spec('netCDF4') is None and importlib.util.find_spec('h5netcdf') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'h5netcdf'])

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import requests, joblib
import xml.etree.ElementTree as ET
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.isotonic import IsotonicRegression
from scipy import ndimage
from pyproj import Transformer, CRS
from dataclasses import dataclass

print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Grid and GeoJSON helpers

Every tile is snapped to the same 1 km lattice in California Albers (EPSG:3310). Row 0 is the north edge.
`grid_id = row * 1000 + col` on that lattice, so the backend can use it as a key.
GeoJSON follows RFC 7946: lon/lat, counter-clockwise rings. Cells already burning are labelled `ACTIVE_FIRE`.

In [ ]:
_to_grid = Transformer.from_crs('EPSG:4326', GRID_CRS, always_xy=True)
_to_wgs = Transformer.from_crs(GRID_CRS, 'EPSG:4326', always_xy=True)


@dataclass(frozen=True)
class Grid:
    x0: float   # west edge (m)
    y0: float   # north edge (m)
    h: int
    w: int
    res: float = RES_M

    @classmethod
    def around_xy(cls, x, y, size=GOES_TILE, res=RES_M):
        c0 = round((x - size * res / 2 - LATTICE_X0) / res)
        r0 = round((LATTICE_Y0 - (y + size * res / 2)) / res)
        return cls(LATTICE_X0 + c0 * res, LATTICE_Y0 - r0 * res, size, size, res)

    @classmethod
    def around(cls, lat, lon, size=GOES_TILE, res=RES_M):
        return cls.around_xy(*_to_grid.transform(lon, lat), size=size, res=res)

    @property
    def bounds(self):
        return self.x0, self.x0 + self.w * self.res, self.y0 - self.h * self.res, self.y0

    def centers(self):
        xs = self.x0 + (np.arange(self.w) + 0.5) * self.res
        ys = self.y0 - (np.arange(self.h) + 0.5) * self.res
        return np.meshgrid(xs, ys)

    def grid_ids(self):
        c0 = round((self.x0 - LATTICE_X0) / self.res)
        r0 = round((LATTICE_Y0 - self.y0) / self.res)
        r, c = np.mgrid[r0:r0 + self.h, c0:c0 + self.w]
        return (r * LATTICE_NCOLS + c).astype(np.int64)


def grid_id_to_center_lonlat(gid, res=RES_M):
    r, c = divmod(int(gid), LATTICE_NCOLS)
    return _to_wgs.transform(LATTICE_X0 + (c + 0.5) * res, LATTICE_Y0 - (r + 0.5) * res)


def risk_label(p, thr):
    return np.where(p >= thr['HIGH_RISK'], 'HIGH_RISK',
                    np.where(p >= thr['MODERATE_RISK'], 'MODERATE_RISK', 'LOW_RISK'))


def to_geojson(p, grid, timestamp, thr, p_emit=P_EMIT, burned=None, extra_layers=None, collection_props=None, ndp=6):
    p = np.asarray(p, float)
    burned = np.zeros(p.shape, bool) if burned is None else np.asarray(burned, bool)
    ids, labels = grid.grid_ids(), risk_label(np.nan_to_num(p), thr)
    cx = grid.x0 + np.arange(grid.w + 1) * grid.res
    cy = grid.y0 - np.arange(grid.h + 1) * grid.res
    LON, LAT = _to_wgs.transform(*np.meshgrid(cx, cy))
    LON, LAT = np.round(LON, ndp), np.round(LAT, ndp)
    feats = []
    rows, cols = np.where((np.isfinite(p) & (p >= p_emit)) | burned)
    for r, c in zip(rows.tolist(), cols.tolist()):
        # SW, SE, NE, NW, SW
        ring = [[LON[r + 1, c], LAT[r + 1, c]], [LON[r + 1, c + 1], LAT[r + 1, c + 1]],
                [LON[r, c + 1], LAT[r, c + 1]], [LON[r, c], LAT[r, c]], [LON[r + 1, c], LAT[r + 1, c]]]
        if burned[r, c]:
            props = {'grid_id': int(ids[r, c]), 'fire_probability': 1.0, 'risk_label': 'ACTIVE_FIRE'}
        else:
            props = {'grid_id': int(ids[r, c]), 'fire_probability': round(float(p[r, c]), 3),
                     'risk_label': str(labels[r, c])}
        for name, layer in (extra_layers or {}).items():
            if layer is not None:
                v = layer[r, c]
                props[name] = None if not np.isfinite(v) else round(float(v), 3)
        feats.append({'type': 'Feature', 'properties': props,
                      'geometry': {'type': 'Polygon', 'coordinates': [[[float(a), float(b)] for a, b in ring]]}})
    fc = {'type': 'FeatureCollection', 'timestamp': timestamp}
    fc.update(collection_props or {})
    fc['features'] = feats
    return fc


def validate_feature_collection(fc):
    ids = [f['properties']['grid_id'] for f in fc['features']]
    assert len(ids) == len(set(ids)), 'duplicate grid_id'
    for f in fc['features']:
        ring = f['geometry']['coordinates'][0]
        assert len(ring) == 5 and ring[0] == ring[-1], 'ring not closed'
        assert sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:])) > 0, 'ring not counter-clockwise'
        pr = f['properties']
        assert 0 <= pr['fire_probability'] <= 1
        assert pr['risk_label'] in {'LOW_RISK', 'MODERATE_RISK', 'HIGH_RISK', 'ACTIVE_FIRE'}
    json.dumps(fc)

## 3. GOES-18 data from NOAA's AWS bucket

The GOES-18 fire product (`ABI-L2-FDCC`) is public on AWS, so no account is needed. Each 2 km GOES pixel is mapped onto the
1 km grid by nearest pixel (same as Earth Engine does for the training data), and the fire mask codes are turned into a
confidence between 0 and 1.

In [ ]:
FIRE_CODES = [10, 30, 11, 31, 12, 32, 13, 33, 14, 34, 15, 35]   # fire classes in the GOES mask
FIRE_CONF  = [1.0, 1.0, 0.9, 0.9, 0.8, 0.8, 0.5, 0.5, 0.3, 0.3, 0.1, 0.1]
FIRE_THRESHOLD = 0.5
_CONF_LUT = np.zeros(256, np.float32)
for _code, _conf in zip(FIRE_CODES, FIRE_CONF):
    _CONF_LUT[_code] = _conf
_S3_NS = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
_GOES_INDEX_CACHE = {}


def _utc(t):
    t = pd.Timestamp(t)
    return t.tz_localize('UTC') if t.tzinfo is None else t.tz_convert('UTC')


def mask_to_conf(mask):
    m = np.clip(np.nan_to_num(np.asarray(mask, float), nan=0.0), 0, 255).astype(int)
    return _CONF_LUT[m]


def _s3_list_keys(bucket, prefix):
    keys, token = [], None
    while True:
        params = {'list-type': '2', 'prefix': prefix}
        if token:
            params['continuation-token'] = token
        r = requests.get(f'https://{bucket}.s3.amazonaws.com/', params=params, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        keys += [e.text for e in root.findall('s3:Contents/s3:Key', _S3_NS)]
        if root.findtext('s3:IsTruncated', default='false', namespaces=_S3_NS) != 'true':
            return keys
        token = root.findtext('s3:NextContinuationToken', namespaces=_S3_NS)


def _s3_download(bucket, key):
    dest = os.path.join(GOES_CACHE, os.path.basename(key))
    if not os.path.exists(dest):
        r = requests.get(f'https://{bucket}.s3.amazonaws.com/{key}', timeout=120)
        r.raise_for_status()
        with open(dest + '.part', 'wb') as f:
            f.write(r.content)
        os.replace(dest + '.part', dest)
    return dest


def goes_keys_for_hour(t_hour):
    t = _utc(t_hour)
    prefix = f'{GOES_PRODUCT}/{t:%Y}/{t.dayofyear:03d}/{t:%H}/'
    return sorted(k for k in _s3_list_keys(GOES_BUCKET, prefix) if k.endswith('.nc'))[::GOES_FILE_STRIDE]


def _goes_indices(ds, grid):
    # row/col of the GOES pixel under each grid cell centre, cached per grid and file geometry
    p = ds['goes_imager_projection'].attrs
    h, lon0 = float(p['perspective_point_height']), float(p['longitude_of_projection_origin'])
    x, y = ds['x'].values.astype(float), ds['y'].values.astype(float)    # scan angles in radians
    key = (grid, h, lon0, x[0], x[-1], len(x), y[0], y[-1], len(y))
    if key not in _GOES_INDEX_CACHE:
        geos = CRS.from_proj4(f"+proj=geos +h={h} +lon_0={lon0} +sweep={p.get('sweep_angle_axis', 'x')} "
                              f"+a={float(p['semi_major_axis'])} +b={float(p['semi_minor_axis'])} +units=m +no_defs")
        X, Y = grid.centers()
        gx, gy = Transformer.from_crs(GRID_CRS, geos, always_xy=True).transform(X, Y)
        ix = np.rint((np.asarray(gx) / h - x[0]) / (x[1] - x[0]))
        iy = np.rint((np.asarray(gy) / h - y[0]) / (y[1] - y[0]))
        ok = np.isfinite(ix) & np.isfinite(iy) & (ix >= 0) & (ix < len(x)) & (iy >= 0) & (iy < len(y))
        _GOES_INDEX_CACHE[key] = (np.where(ok, iy, 0).astype(int), np.where(ok, ix, 0).astype(int), ok)
    return _GOES_INDEX_CACHE[key]


def read_fdc_on_grid(path, grid):
    """One GOES file -> fire confidence, clear-sky look (0/1) and fire power (MW) on the grid."""
    import xarray as xr
    with xr.open_dataset(path) as ds:
        iy, ix, ok = _goes_indices(ds, grid)
        mask, dqf, power = ds['Mask'].values, ds['DQF'].values, ds['Power'].values
    conf = np.where(ok, mask_to_conf(mask[iy, ix]), 0.0)
    d = np.asarray(dqf[iy, ix], float)
    clear = ok & np.isfinite(d) & (d <= 1)     # DQF 0 = good fire pixel, 1 = good fire-free land
    frp = np.where(ok, np.nan_to_num(np.asarray(power[iy, ix], float), nan=0.0), 0.0)
    return conf.astype(np.float32), clear.astype(np.float32), frp.astype(np.float32)


def goes_aws_hourly(grid, t0, n_hours):
    """Hourly max fire confidence, number of clear looks and max fire power, each [h, w, n_hours]."""
    t0 = _utc(t0).floor('h')
    conf = np.zeros((grid.h, grid.w, n_hours), np.float32)
    nval, frp = np.zeros_like(conf), np.zeros_like(conf)
    for k in range(n_hours):
        keys = goes_keys_for_hour(t0 + pd.Timedelta(hours=k))
        if not keys:
            print(f'  no GOES files for {t0 + pd.Timedelta(hours=k):%Y-%m-%d %H}Z')
        for key in keys:
            c, v, f = read_fdc_on_grid(_s3_download(GOES_BUCKET, key), grid)
            conf[..., k] = np.maximum(conf[..., k], c)
            nval[..., k] += v
            frp[..., k] = np.maximum(frp[..., k], f)
    return conf, nval, frp

## 4. Current fire layer

Cells where GOES-18 saw fire (confidence 0.5 or more) in the hour before the forecast time. This is the
"Current Fire Detection" layer on the map. It needs no model.

In [ ]:
GOES_T = (pd.Timestamp.now('UTC').floor('h') if GOES_TIME == 'now'
          else _utc(GOES_TIME or EVENT['time']).floor('h'))
if GOES_CENTER is not None:
    GOES_GRID = Grid.around(*GOES_CENTER)
else:
    _b = EVENT['pinn_bounds']
    GOES_GRID = Grid.around_xy((_b[0] + _b[1]) / 2, (_b[2] + _b[3]) / 2)
print('forecast time:', GOES_T, '| tile:', GOES_GRID)


def current_fire_geojson(grid, t_end, hours=1):
    t_end = _utc(t_end)
    conf, _, frp = goes_aws_hourly(grid, t_end - pd.Timedelta(hours=hours), hours)
    c, f = conf.max(-1), frp.max(-1)
    fmt = '%Y-%m-%dT%H:%M:%SZ'
    return to_geojson(np.zeros_like(c), grid, t_end.strftime(fmt), {'MODERATE_RISK': 1, 'HIGH_RISK': 1},
                      p_emit=2.0, burned=c >= FIRE_THRESHOLD, extra_layers={'fire_confidence': c, 'frp_mw': f},
                      collection_props={'layer': 'current_fire_detection', 'source': f'GOES-18 {GOES_PRODUCT}',
                                        'window_start': (t_end - pd.Timedelta(hours=hours)).strftime(fmt),
                                        'window_end': t_end.strftime(fmt)})


if RUN_GOES_CURRENT_FIRE:
    try:
        FC_CURRENT = current_fire_geojson(GOES_GRID, GOES_T)
        validate_feature_collection(FC_CURRENT)
        CURRENT_PATH = os.path.join(OUTPUT_DIR, f'current_fire_goes18_{GOES_T:%Y%m%dT%H}Z.geojson')
        json.dump(FC_CURRENT, open(CURRENT_PATH, 'w'))
        print(f"{len(FC_CURRENT['features'])} fire cells -> {CURRENT_PATH}")
    except requests.RequestException as ex:
        print('could not reach the NOAA bucket:', ex)

## 5. Training samples (Earth Engine)

For each large California fire in MTBS, a 96 km tile is centred on the burn scar and a sample is saved every 3 hours while
GOES sees fire. Inputs (12 channels): fire in the last hour, fire 1-4 h ago, fire in the last 24 h, temperature, humidity,
wind u/v, gust, rain, elevation, slope, NDVI. Labels: fire seen in the next 1, 3 and 6 hours (1), no fire (0), or never
seen clearly because of cloud or smoke (-1, ignored in training).

The build takes 1-3 hours and can be re-run after a disconnect: finished fires are skipped.

In [ ]:
GOES_WEST = [('NOAA/GOES/17/FDCC', '2019-01-01', '2023-01-04'),
             ('NOAA/GOES/18/FDCC', '2023-01-04', '2100-01-01')]
RTMA_BANDS = ['TMP', 'DPT', 'UGRD', 'VGRD', 'GUST', 'ACPC01']
GOES_CHANNELS = ['fire_now', 'fire_prev3h', 'fire_24h', 'tmp_c', 'rh', 'wind_u', 'wind_v', 'gust', 'precip',
                 'elevation', 'slope', 'NDVI']
GOES_FIRE_CHANNELS = 3      # the first 3 channels are already 0-1 and are not normalized
GOES_CHUNK_H = 24


def _pixels(img, grid):
    arr = ee.data.computePixels({
        'expression': img.toFloat(), 'fileFormat': 'NUMPY_NDARRAY',
        'grid': {'dimensions': {'width': grid.w, 'height': grid.h},
                 'affineTransform': {'scaleX': grid.res, 'shearX': 0, 'translateX': grid.x0,
                                     'shearY': 0, 'scaleY': -grid.res, 'translateY': grid.y0},
                 'crsCode': GRID_CRS}})
    out = {n: np.asarray(arr[n], np.float32) for n in arr.dtype.names}
    return {n: np.where(np.isfinite(a), a, np.nan) for n, a in out.items()}   # masked pixels come back as -inf


def _goes_collection(start, end):
    cols = [ee.ImageCollection(cid).filterDate(max(start, s), min(end, e))
            for cid, s, e in GOES_WEST if start < e and end > s]
    out = cols[0]
    for c_ in cols[1:]:
        out = out.merge(c_)
    return out


def goes_ee_hourly(grid, t0, n_hours):
    """Same as goes_aws_hourly but from Earth Engine (used for training)."""
    conf, nval = [], []
    t0 = _utc(t0).floor('h')
    fmt = lambda t: t.strftime('%Y-%m-%dT%H:%M:%S')
    for c0 in range(0, n_hours, GOES_CHUNK_H):
        n = min(GOES_CHUNK_H, n_hours - c0)
        start = t0 + pd.Timedelta(hours=c0)
        coll = _goes_collection(fmt(start), fmt(start + pd.Timedelta(hours=n)))
        e0 = ee.Date(fmt(start))

        def one_hour(i):
            s = e0.advance(i, 'hour')
            col = coll.filterDate(s, s.advance(1, 'hour'))
            empty = col.size().eq(0)
            cf = ee.Image(ee.Algorithms.If(empty, ee.Image.constant(0),
                                           col.map(lambda im: im.select('Mask').remap(FIRE_CODES, FIRE_CONF, 0)).max()))
            vl = ee.Image(ee.Algorithms.If(empty, ee.Image.constant(0),
                                           col.map(lambda im: im.select('DQF').lte(1)).sum()))
            return cf.rename('conf').unmask(0).addBands(vl.rename('nvalid').unmask(0)).toFloat()

        px = _pixels(ee.ImageCollection(ee.List.sequence(0, n - 1).map(one_hour)).toBands(), grid)
        conf += [np.nan_to_num(px[f'{i}_conf']) for i in range(n)]
        nval += [np.nan_to_num(px[f'{i}_nvalid']) for i in range(n)]
    return np.stack(conf, -1), np.stack(nval, -1)


def rtma_at(grid, t):
    e = ee.Date(_utc(t).strftime('%Y-%m-%dT%H:%M:%S'))
    img = (ee.ImageCollection('NOAA/NWS/RTMA').filterDate(e.advance(-2, 'hour'), e.advance(1, 'minute'))
             .sort('system:time_start', False).first())
    return _pixels(ee.Image(img).select(RTMA_BANDS), grid)


def static_layers(grid, date):
    dem = ee.Image('USGS/SRTMGL1_003')
    d = ee.Date(_utc(date).strftime('%Y-%m-%d'))
    ndvi = (ee.ImageCollection('NASA/VIIRS/002/VNP13A1').filterDate(d.advance(-40, 'day'), d)
              .sort('system:time_start', False).first())
    return _pixels(ee.Image.cat([dem.select('elevation'), ee.Terrain.slope(dem).rename('slope'),
                                 ee.Image(ndvi).select('NDVI')]), grid)


def relative_humidity(t_c, td_c):
    return np.clip(100.0 * np.exp(17.625 * td_c / (243.04 + td_c)) / np.exp(17.625 * t_c / (243.04 + t_c)), 0, 100)


def window(conf, nval, a, b):
    return conf[..., a:b].max(-1), nval[..., a:b].sum(-1)


def make_label(conf_w, nval_w):
    return np.where(conf_w >= FIRE_THRESHOLD, 1.0, np.where(nval_w > 0, 0.0, -1.0)).astype(np.float32)


def goes_inputs(conf, nval, k, weather, static):
    # k is the issue hour in the stack; hour k-1 is "the last hour"
    now_c, _ = window(conf, nval, k - 1, k)
    prev_c, _ = window(conf, nval, max(k - 4, 0), k - 1)
    day_c, _ = window(conf, nval, max(k - 24, 0), k)
    w = weather
    x = np.stack([(now_c >= FIRE_THRESHOLD).astype(np.float32), prev_c, day_c,
                  w['TMP'], relative_humidity(w['TMP'], w['DPT']), w['UGRD'], w['VGRD'], w['GUST'], w['ACPC01'],
                  static['elevation'], static['slope'], static['NDVI']], -1)
    return x.astype(np.float32)


def samples_for_event(conf, nval, t_start, weather_fn, static, step_h=GOES_STEP_H, horizons=GOES_HORIZONS_H):
    for k in range(24, conf.shape[-1] - max(horizons), step_h):
        if (window(conf, nval, k - 1, k)[0] >= FIRE_THRESHOLD).sum() == 0:
            continue
        t = _utc(t_start) + pd.Timedelta(hours=k - 24)
        try:
            w = weather_fn(t)
        except Exception as ex:
            print(f'  skip {t:%Y-%m-%d %H}Z, no weather ({str(ex)[:60]})')
            continue
        yield t, goes_inputs(conf, nval, k, w, static), {h: make_label(*window(conf, nval, k, k + h)) for h in horizons}


def _parse_mtbs_date(v):
    v = int(v)
    if 19840101 <= v <= 21001231:
        return pd.to_datetime(str(v), format='%Y%m%d', utc=True)
    return pd.to_datetime(v, unit='ms', utc=True)


def mtbs_fire_list():
    ca = ee.Geometry.Rectangle(list(CA_BBOX))
    fc = (ee.FeatureCollection('USFS/GTAC/MTBS/burned_area_boundaries/v1').filterBounds(ca)
            .filter(ee.Filter.gte('BurnBndAc', GOES_MIN_ACRES))
            .map(lambda f: ee.Feature(f.geometry().centroid(100),
                                      {'name': f.get('Incid_Name'), 'ig': f.get('Ig_Date'), 'acres': f.get('BurnBndAc')})))
    rows = []
    for f in fc.getInfo()['features']:
        p, (lon, lat) = f['properties'], f['geometry']['coordinates']
        start = _parse_mtbs_date(p['ig'])
        if GOES_YEARS[0] <= start.year <= GOES_YEARS[1]:
            name = re.sub(r'[^A-Za-z0-9]+', '_', str(p['name'])).strip('_')
            rows.append({'name': f'{name}_{start.year}', 'lat': lat, 'lon': lon, 'start_utc': start,
                         'acres': p['acres'], 'year': start.year})
    df = pd.DataFrame(rows).drop_duplicates('name')
    df = df.sort_values('acres', ascending=False).groupby('year').head(GOES_MAX_PER_YEAR).sort_values('start_utc')
    df['end_utc'] = df['start_utc'] + pd.Timedelta(days=GOES_EVENT_DAYS)
    return df.reset_index(drop=True)


def build_event(name, lat, lon, start_utc, end_utc, out_dir):
    done = os.path.join(out_dir, f'{name}.done')
    if os.path.exists(done):
        return 0
    grid = Grid.around(lat, lon)
    t0 = _utc(start_utc) - pd.Timedelta(hours=24)
    n_hours = int((_utc(end_utc) - t0) / pd.Timedelta(hours=1)) + max(GOES_HORIZONS_H)
    conf, nval = goes_ee_hourly(grid, t0, n_hours)
    static = static_layers(grid, start_utc)
    n = 0
    for t, x, ys in samples_for_event(conf, nval, start_utc, lambda t: rtma_at(grid, t), static):
        np.savez_compressed(os.path.join(out_dir, f'{name}_{t:%Y%m%dT%H}.npz'), x=x, fire=name, t=t.isoformat(),
                            **{f'y_h{h}': y for h, y in ys.items()})
        n += 1
    open(done, 'w').write(str(n))
    return n


def load_goes_samples(out_dir, horizon_h):
    files = sorted(glob.glob(os.path.join(out_dir, '*.npz')))
    if not files:
        raise FileNotFoundError(f'no samples in {out_dir}, run the build first')
    X, Y, F, T = [], [], [], []
    for f in files:
        d = np.load(f)
        X.append(np.where(np.isfinite(d['x']), d['x'], np.nan))    # older samples stored masked pixels as -inf
        Y.append(d[f'y_h{horizon_h}'][..., None])
        F.append(str(d['fire']))
        T.append(str(d['t']))
    return np.stack(X), np.stack(Y), np.array(F), pd.to_datetime(T, utc=True)

In [ ]:
if RUN_GOES_BUILD:
    os.makedirs(GOES_DATA_DIR, exist_ok=True)
    FIRES = mtbs_fire_list()
    FIRES.to_csv(os.path.join(GOES_DATA_DIR, 'fires.csv'), index=False)
    display(FIRES[['name', 'start_utc', 'acres', 'lat', 'lon']])
    for r in FIRES.itertuples():
        try:
            print(f'{r.name}: {build_event(r.name, r.lat, r.lon, r.start_utc, r.end_utc, GOES_DATA_DIR)} samples')
        except Exception as ex:
            print(f'{r.name}: failed ({str(ex)[:120]}), re-run later')
    print('samples:', len(glob.glob(os.path.join(GOES_DATA_DIR, '*.npz'))))

## 6. Model and metrics

A standard U-Net (3 down/up levels, 32 base filters). It is fully convolutional, so it trains on 32x32 crops and predicts
on full 96x96 tiles. Loss is binary cross-entropy with fire pixels weighted 10x; cloudy pixels (-1) are ignored.

Metrics are compared against three simple baselines:

- persistence: fire stays where it is now
- dilation: current fire grown by one cell in every direction
- distance: risk falls off with distance to the current fire

"New fire" means cells that are not burning now. That is the part that actually tests spread.

In [ ]:
def build_unet(input_shape, num_filters_start=FILTERS):
    inputs = layers.Input(shape=input_shape)

    def block(x, f, drop=None):
        x = layers.Conv2D(f, 3, activation='relu', padding='same')(x)
        x = layers.Conv2D(f, 3, activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        return layers.Dropout(drop)(x) if drop else x

    f = num_filters_start
    c1 = layers.BatchNormalization()(layers.Conv2D(f, 3, activation='relu', padding='same')(
        layers.Conv2D(f, 3, activation='relu', padding='same')(inputs)))
    c2 = block(layers.Dropout(0.2)(layers.MaxPooling2D(2)(c1)), f * 2)
    c3 = block(layers.Dropout(0.2)(layers.MaxPooling2D(2)(c2)), f * 4)
    c4 = block(layers.Dropout(0.3)(layers.MaxPooling2D(2)(c3)), f * 8, drop=0.3)
    c5 = block(layers.concatenate([layers.Conv2DTranspose(f * 4, 2, strides=2, padding='same')(c4), c3]), f * 4)
    c6 = block(layers.concatenate([layers.Conv2DTranspose(f * 2, 2, strides=2, padding='same')(c5), c2]), f * 2)
    c7 = block(layers.concatenate([layers.Conv2DTranspose(f, 2, strides=2, padding='same')(c6), c1]), f)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return Model(inputs=[inputs], outputs=[outputs])


def masked_weighted_bce(pos_weight=POS_WEIGHT):
    def loss(y_true, y_pred):
        valid = tf.cast(y_true >= 0, tf.float32)
        y = tf.clip_by_value(y_true, 0.0, 1.0)
        p = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        l = -(pos_weight * y * tf.math.log(p) + (1 - y) * tf.math.log(1 - p))
        return tf.reduce_sum(l * valid) / (tf.reduce_sum(valid) + 1e-6)
    return loss


class MaskedAUC(tf.keras.metrics.AUC):
    def update_state(self, y_true, y_pred, sample_weight=None):
        valid = tf.cast(y_true >= 0, y_pred.dtype)
        return super().update_state(tf.clip_by_value(y_true, 0.0, 1.0), y_pred, sample_weight=valid)


def make_model(n_channels):
    m = build_unet((None, None, n_channels))
    m.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=masked_weighted_bce(),
              metrics=[MaskedAUC(curve='PR', name='pr_auc'), MaskedAUC(curve='ROC', name='roc_auc')])
    return m


def best_f1_threshold(y, p):
    m = y >= 0
    prec, rec, thr = precision_recall_curve(y[m], p[m])
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thr[np.argmax(f1[:-1])])


def score(y, p, thr, subset=None):
    m = (y >= 0) if subset is None else ((y >= 0) & subset)
    yy, pp = (y[m] == 1), p[m]
    yhat = pp >= thr
    tp, fp, fn = np.sum(yhat & yy), np.sum(yhat & ~yy), np.sum(~yhat & yy)
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    out = {'n_pos': int(yy.sum()), 'precision': prec, 'recall': rec,
           'f1': 2 * prec * rec / max(prec + rec, 1e-9), 'threshold': thr}
    if len(np.unique(pp)) > 2 and 0 < yy.sum() < yy.size:
        out['pr_auc'] = average_precision_score(yy, pp)
        out['roc_auc'] = roc_auc_score(yy, pp)
    return out


def fire_distance_km(prev, cap=MAX_FIRE_DIST_KM):
    out = np.full(prev.shape, float(cap), np.float32)
    for i in range(len(prev)):
        fire = prev[i] > 0
        if fire.any():
            out[i] = np.minimum(ndimage.distance_transform_edt(~fire), cap)
    return out


def dilation_baseline(prev, px=1):
    return ndimage.binary_dilation(prev > 0, structure=np.ones((1, 2 * px + 1, 2 * px + 1), bool)).astype(np.float32)


def fit_isotonic(Y, P):
    # calibration so that "30%" means fire about 30% of the time; negatives subsampled to keep it fast
    m = Y >= 0
    p, y = P[m], (Y[m] == 1).astype(float)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n_neg = min(len(neg), 50 * max(len(pos), 1))
    sel = np.concatenate([pos, np.random.default_rng(0).choice(neg, size=n_neg, replace=False)])
    w = np.where(y[sel] == 1, 1.0, len(neg) / max(n_neg, 1))
    return IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0).fit(p[sel], y[sel], sample_weight=w)


def derive_risk_thresholds(y, p_cal, target_recall=MODERATE_TARGET_RECALL):
    # HIGH = best F1 on validation, MODERATE = lowest cut that still catches target_recall of fire pixels
    m = y >= 0
    prec, rec, thr = precision_recall_curve(y[m] == 1, p_cal[m])
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    high = min(float(thr[np.argmax(f1[:-1])]), P_MAX)
    ok = np.where(rec[:-1] >= target_recall)[0]
    moderate = max(float(thr[ok.max()]) if len(ok) else float(thr[0]), P_EMIT)
    if moderate >= high:
        moderate = high / 2
    return {'MODERATE_RISK': moderate, 'HIGH_RISK': high}

## 7. Train the +1, +3 and +6 hour models

One model per horizon. Fires are split by date: the newest 20% are the test set and the 15% before them are validation,
so no fire appears in two sets. Each model is saved to `unet_bundle_v3/goes_h{1,3,6}/` with its normalization,
calibration and risk thresholds.

In [ ]:
def chrono_fire_split(F, T, val_frac=0.15, test_frac=0.20):
    first = pd.Series(T, index=F).groupby(level=0).min().sort_values()
    fires = list(first.index)
    assert len(fires) >= 3, f'only {len(fires)} fires with samples, need at least 3'
    n_test = max(1, int(round(test_frac * len(fires))))
    n_val = max(1, int(round(val_frac * len(fires))))
    test_f, val_f = fires[-n_test:], fires[-(n_test + n_val):-n_test]
    return ~np.isin(F, test_f + val_f), np.isin(F, val_f), np.isin(F, test_f), val_f, test_f


def goes_norm_stats(X):
    mu = np.nanmean(X, axis=(0, 1, 2))
    sd = np.maximum(np.nanstd(X, axis=(0, 1, 2)), 1e-3)
    mu[:GOES_FIRE_CHANNELS], sd[:GOES_FIRE_CHANNELS] = 0.0, 1.0
    return mu.astype(np.float32), sd.astype(np.float32)


def goes_normalize(X, mu, sd):
    bad = ~(np.isfinite(mu) & np.isfinite(sd))       # bundles trained before the -inf fix have no stats for rain/NDVI
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (np.asarray(X, np.float32) - np.where(bad, 0, mu)) / np.where(bad, 1, sd)
    z[..., bad] = 0.0
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0), -10, 10).astype(np.float32)


def goes_tf_dataset(X, Y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if training:
        c = X.shape[-1]
        crop = lambda x, y: tuple(tf.split(tf.image.random_crop(tf.concat([x, y], -1), [TRAIN_CROP, TRAIN_CROP, c + 1]),
                                           [c, 1], axis=-1))
        ds = ds.shuffle(len(X)).map(crop, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def train_goes_horizon(h):
    X, Y, F, T = load_goes_samples(GOES_DATA_DIR, h)
    tr, va, te, val_f, test_f = chrono_fire_split(F, T)
    print(f'+{h} h: {len(X)} samples, {len(set(F))} fires (train {tr.sum()}, val {va.sum()}, test {te.sum()})')
    mu, sd = goes_norm_stats(X[tr])
    Xn = goes_normalize(X, mu, sd)
    tf.keras.utils.set_random_seed(0)
    model = make_model(Xn.shape[-1])
    ckpt = os.path.join(OUTPUT_DIR, f'goes_h{h}.weights.h5')
    model.fit(goes_tf_dataset(Xn[tr], Y[tr], True), validation_data=goes_tf_dataset(Xn[va], Y[va], False),
              epochs=EPOCHS, verbose=2,
              callbacks=[ModelCheckpoint(ckpt, monitor='val_pr_auc', mode='max', save_best_only=True, save_weights_only=True),
                         EarlyStopping(monitor='val_pr_auc', mode='max', patience=PATIENCE)])
    model.load_weights(ckpt)
    Pv = model.predict(Xn[va], batch_size=16, verbose=0)[..., 0]
    Pt = model.predict(Xn[te], batch_size=16, verbose=0)[..., 0]
    Yv, Yt, prev_t = Y[va][..., 0], Y[te][..., 0], X[te][..., 0]
    cal = fit_isotonic(Yv, Pv)
    calib = lambda p: np.minimum(cal.predict(p.ravel()).reshape(p.shape), P_MAX)
    thr = derive_risk_thresholds(Yv, calib(Pv))

    dist_thr = best_f1_threshold(Yv, np.exp(-fire_distance_km(X[va][..., 0])))
    candidates = {'U-Net': (calib(Pt), thr['HIGH_RISK']),
                  'Persistence': ((prev_t > 0).astype(np.float32), 0.5),
                  'Dilation 1 px': (dilation_baseline(prev_t), 0.5),
                  'Distance': (np.exp(-fire_distance_km(prev_t)), dist_thr)}
    rows = [{'model': n, 'pixels': s, **score(Yt, p, t_, sub)}
            for n, (p, t_) in candidates.items() for s, sub in [('all', None), ('new fire', prev_t == 0)]]
    table = pd.DataFrame(rows)
    display(table.pivot(index='model', columns='pixels', values=['pr_auc', 'f1', 'precision', 'recall']).round(3))

    bdir = os.path.join(BUNDLE_DIR, f'goes_h{h}')
    os.makedirs(bdir, exist_ok=True)
    export = build_unet((None, None, Xn.shape[-1]))
    export.set_weights(model.get_weights())
    export.save(os.path.join(bdir, 'model.keras'))
    json.dump({'channels': GOES_CHANNELS, 'mu': mu.tolist(), 'sd': sd.tolist()}, open(os.path.join(bdir, 'norm.json'), 'w'))
    joblib.dump(cal, os.path.join(bdir, 'calibrator.joblib'))
    json.dump(thr, open(os.path.join(bdir, 'thresholds.json'), 'w'), indent=1)
    new = table.query("model == 'U-Net' and pixels == 'new fire'").iloc[0]
    json.dump({'created_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
               'model_version': f'unet-goes-h{h}-v1', 'horizon_hours': h,
               'target': f'probability that GOES sees fire in the 1 km cell within {h} h',
               'train_fires': sorted(set(F[tr])), 'val_fires': val_f, 'test_fires': test_f,
               'test_new_fire_pr_auc': float(new.get('pr_auc', np.nan))},
              open(os.path.join(bdir, 'manifest.json'), 'w'), indent=1)
    table.to_csv(os.path.join(OUTPUT_DIR, f'goes_h{h}_test_table.csv'), index=False)
    print('saved', bdir, {k: round(v, 3) for k, v in thr.items()})
    return table


if RUN_GOES_TRAIN:
    GOES_TABLES = {h: train_goes_horizon(h) for h in GOES_HORIZONS_H}

## 8. Forecast inputs

At forecast time the fire inputs come from the last 24 hours of GOES-18 on AWS, weather from Open-Meteo (no key needed),
and terrain/NDVI from Earth Engine. Open-Meteo is sampled on a 9x9 grid and interpolated to the tile.

If the PINN checkpoint and its settings are available, its front is turned into a probability and averaged with the U-Net.

In [ ]:
OPEN_METEO_VARS = ['temperature_2m', 'dew_point_2m', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
                   'precipitation']


def _open_meteo_get(url, params):
    r = requests.get(url, params=params, timeout=90)
    r.raise_for_status()
    return r.json()


def openmeteo_weather(grid, t, n=WEATHER_GRID_N):
    from scipy.interpolate import RegularGridInterpolator
    t = _utc(t).floor('h')
    xmin, xmax, ymin, ymax = grid.bounds
    xs, ys = np.linspace(xmin, xmax, n), np.linspace(ymin, ymax, n)
    lon, lat = _to_wgs.transform(*np.meshgrid(xs, ys))
    recent = pd.Timestamp.now('UTC') - t < pd.Timedelta(days=80)
    url = ('https://api.open-meteo.com/v1/forecast' if recent
           else 'https://historical-forecast-api.open-meteo.com/v1/forecast')
    js = _open_meteo_get(url, {'latitude': ','.join(f'{v:.4f}' for v in np.ravel(lat)),
                               'longitude': ','.join(f'{v:.4f}' for v in np.ravel(lon)),
                               'hourly': ','.join(OPEN_METEO_VARS), 'wind_speed_unit': 'ms', 'timezone': 'UTC',
                               'start_date': f'{t:%Y-%m-%d}', 'end_date': f'{t:%Y-%m-%d}'})
    js = js if isinstance(js, list) else [js]
    key = t.strftime('%Y-%m-%dT%H:00')
    v = {name: np.array([loc['hourly'][name][loc['hourly']['time'].index(key)] for loc in js], float).reshape(n, n)
         for name in OPEN_METEO_VARS}
    wd = np.deg2rad(v['wind_direction_10m'])
    fields = {'TMP': v['temperature_2m'], 'DPT': v['dew_point_2m'],
              'UGRD': -v['wind_speed_10m'] * np.sin(wd), 'VGRD': -v['wind_speed_10m'] * np.cos(wd),
              'GUST': v['wind_gusts_10m'], 'ACPC01': v['precipitation']}
    X, Y = grid.centers()
    pts = np.c_[Y.ravel(), X.ravel()]
    return {k: RegularGridInterpolator((ys, xs), np.nan_to_num(f), bounds_error=False, fill_value=None)(pts)
               .reshape(X.shape).astype(np.float32) for k, f in fields.items()}


def load_goes_bundle(h):
    bdir = os.path.join(BUNDLE_DIR, f'goes_h{h}')
    norm = json.load(open(os.path.join(bdir, 'norm.json')))
    b = {'model': tf.keras.models.load_model(os.path.join(bdir, 'model.keras'), compile=False),
         'mu': np.array(norm['mu'], np.float32), 'sd': np.array(norm['sd'], np.float32),
         'calibrator': joblib.load(os.path.join(bdir, 'calibrator.joblib')),
         'thresholds': json.load(open(os.path.join(bdir, 'thresholds.json'))),
         'manifest': json.load(open(os.path.join(bdir, 'manifest.json')))}
    bad = [GOES_CHANNELS[i] for i in np.where(~(np.isfinite(b['mu']) & np.isfinite(b['sd'])))[0]]
    if bad:
        print(f'+{h} h model was trained without {bad}; retrain (Section 7) to use them')
    return b


def pinn_prob(phi_m, grid, t_valid, s_m, k=3, inside_negative=True, bounds=None):
    # PINN signed distance (m) -> probability: share of each cell inside the predicted front
    X, Y = grid.centers()
    off = ((np.arange(k) + 0.5) / k - 0.5) * grid.res
    xs = np.broadcast_to(X[..., None, None] + off[None, None, None, :], X.shape + (k, k))
    ys = np.broadcast_to(Y[..., None, None] - off[None, None, :, None], X.shape + (k, k))
    phi = np.asarray(phi_m(xs.ravel(), ys.ravel(), t_valid), float).reshape(xs.shape)
    z = (-phi if inside_negative else phi) / s_m
    p = (1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))).mean(axis=(-2, -1))
    if bounds is not None:
        xmin, xmax, ymin, ymax = bounds
        p[~((X >= xmin) & (X <= xmax) & (Y >= ymin) & (Y <= ymax))] = np.nan
    return p


def fuse(p_unet, p_pinn=None, w_unet=W_UNET):
    p = np.asarray(p_unet, float).copy()
    if p_pinn is not None:
        both = np.isfinite(p_pinn) & np.isfinite(p)
        p[both] = w_unet * p[both] + (1.0 - w_unet) * p_pinn[both]
    return np.clip(p, 0.0, 1.0)


def load_pinn_phi(cfg):
    missing = [k for k in ('x0', 'y0', 'L_m') if cfg[k] is None]
    if not os.path.exists(cfg['checkpoint']) or missing:
        print('no PINN, running U-Net only')
        return None
    import torch, torch.nn as nn
    obj = torch.load(cfg['checkpoint'], map_location='cpu', weights_only=False)
    if isinstance(obj, nn.Module):
        net = obj
    else:
        state = obj.get('state_dict', obj.get('model', obj)) if isinstance(obj, dict) else obj
        mods, d = [], 3
        for _ in range(cfg['layers']):
            mods += [nn.Linear(d, cfg['hidden']), nn.Tanh()]
            d = cfg['hidden']
        net = nn.Sequential(*mods, nn.Linear(d, 1))
        try:
            net.load_state_dict(state)
        except RuntimeError:     # different key names: copy Linear layers in order
            lin = [mod for mod in net if isinstance(mod, nn.Linear)]
            ws = [v for k, v in state.items() if k.endswith('weight')]
            bs = [v for k, v in state.items() if k.endswith('bias')]
            assert len(ws) == len(bs) == len(lin)
            with torch.no_grad():
                for mod, w_, b_ in zip(lin, ws, bs):
                    mod.weight.copy_(w_); mod.bias.copy_(b_)
    net.eval()
    t0 = pd.Timestamp(cfg['t_origin_utc'])

    def phi_m(x, y, t_utc):
        tau = (pd.Timestamp(t_utc) - t0).total_seconds() / 86400.0 / cfg['T_days']
        inp = np.stack([np.full(np.shape(x), tau), (np.asarray(x) - cfg['x0']) / cfg['L_m'],
                        (np.asarray(y) - cfg['y0']) / cfg['L_m']], axis=-1).astype(np.float32)
        with torch.no_grad():
            return cfg['L_m'] * net(torch.from_numpy(inp)).numpy()[:, 0]
    return phi_m


def goes_forecast(grid, t, bundles, phi=None, pinn_bounds=None):
    t = _utc(t).floor('h')
    conf, nval, _ = goes_aws_hourly(grid, t - pd.Timedelta(hours=24), 24)
    x = goes_inputs(conf, nval, 24, openmeteo_weather(grid, t), static_layers(grid, t))
    burned = window(conf, nval, 0, 24)[0] >= FIRE_THRESHOLD
    out = {}
    for h, b in bundles.items():
        p = b['model'](goes_normalize(x, b['mu'], b['sd'])[None], training=False).numpy()[0, :, :, 0]
        p = np.minimum(b['calibrator'].predict(p.ravel()).reshape(p.shape), P_MAX)
        p_pinn = (pinn_prob(phi, grid, t + pd.Timedelta(hours=h), PINN_CFG['s_m'],
                            inside_negative=PINN_CFG['inside_negative'], bounds=pinn_bounds) if phi else None)
        out[h] = (p, p_pinn, fuse(p, p_pinn))
    return x, burned, out


def export_netcdf(out, grid, t, path):
    # all horizons in one file, for the PINN team (same 1 km grid as the geojson)
    import xarray as xr
    X, Y = grid.centers()
    data = {f'p_h{h}': (('y', 'x'), np.asarray(p_f, np.float32)) for h, (_, _, p_f) in out.items()}
    data['grid_id'] = (('y', 'x'), grid.grid_ids().astype(np.int32))
    ds = xr.Dataset(data, coords={'x': ('x', X[0, :]), 'y': ('y', Y[:, 0])})
    ds.attrs.update({'crs': GRID_CRS, 'issued_at': t.isoformat(), 'resolution_m': grid.res,
                     'horizons_h': ','.join(str(h) for h in out)})
    ds.to_netcdf(path)

## 9. Forecast and export

Writes one GeoJSON per horizon (`spread_goes_<time>_h1/h3/h6.geojson`) plus a NetCDF with all horizons.
When the forecast time is in the past (the Palisades replay), the next 6 hours of GOES are downloaded to score it.

In [ ]:
if RUN_GOES_FORECAST:
    GOES_BUNDLES = {h: load_goes_bundle(h) for h in GOES_HORIZONS_H
                    if os.path.exists(os.path.join(BUNDLE_DIR, f'goes_h{h}', 'model.keras'))}
    assert GOES_BUNDLES, 'no trained models yet, run Sections 5 and 7 first'
    PHI = load_pinn_phi(PINN_CFG)
    PINN_BOUNDS = EVENT['pinn_bounds'] if PHI else None
    GX, GOES_BURNED, GOES_OUT = goes_forecast(GOES_GRID, GOES_T, GOES_BUNDLES, PHI, PINN_BOUNDS)

    fmt = '%Y-%m-%dT%H:%M:%SZ'
    GOES_PATHS = {}
    for h, (p_u, p_p, p_f) in GOES_OUT.items():
        thr = GOES_BUNDLES[h]['thresholds']
        fc = to_geojson(p_f, GOES_GRID, GOES_T.strftime(fmt), thr, burned=GOES_BURNED,
                        extra_layers={'p_unet': p_u, 'p_pinn': p_p},
                        collection_props={'valid_time': (GOES_T + pd.Timedelta(hours=h)).strftime(fmt),
                                          'horizon_hours': h, 'layer': 'spread_risk',
                                          'model_version': GOES_BUNDLES[h]['manifest']['model_version'],
                                          'members': ['unet-goes'] + (['pinn'] if p_p is not None else []),
                                          'risk_thresholds': thr})
        validate_feature_collection(fc)
        GOES_PATHS[h] = os.path.join(OUTPUT_DIR, f'spread_goes_{GOES_T:%Y%m%dT%H}Z_h{h}.geojson')
        json.dump(fc, open(GOES_PATHS[h], 'w'))
        labels = pd.Series([f['properties']['risk_label'] for f in fc['features']]).value_counts().to_dict()
        print(f'+{h} h: {len(fc["features"])} cells {labels}')
    NC_PATH = os.path.join(OUTPUT_DIR, f'spread_goes_{GOES_T:%Y%m%dT%H}Z.nc')
    export_netcdf(GOES_OUT, GOES_GRID, GOES_T, NC_PATH)
    print('saved to', OUTPUT_DIR)

In [ ]:
# score against what GOES actually saw (only possible for past forecast times)
if RUN_GOES_FORECAST and GOES_T + pd.Timedelta(hours=max(GOES_OUT)) < pd.Timestamp.now('UTC'):
    fconf, fnval, _ = goes_aws_hourly(GOES_GRID, GOES_T, max(GOES_OUT))
    prev = GX[..., 0]
    rows = []
    for h, (_, _, p_f) in GOES_OUT.items():
        y = make_label(*window(fconf, fnval, 0, h))
        for name, sub in [('all', None), ('new fire', prev == 0)]:
            rows.append({'horizon_h': h, 'pixels': name, **score(y, p_f, GOES_BUNDLES[h]['thresholds']['HIGH_RISK'], sub)})
    GOES_EVENT_SCORES = pd.DataFrame(rows)
    cols = ['horizon_h', 'pixels', 'n_pos', 'precision', 'recall', 'f1'] + [c for c in ['pr_auc'] if c in GOES_EVENT_SCORES]
    display(GOES_EVENT_SCORES[cols].round(3))

In [ ]:
if RUN_GOES_FORECAST:
    fig, axes = plt.subplots(1, len(GOES_OUT) + 1, figsize=(5 * (len(GOES_OUT) + 1), 4.6))
    axes[0].imshow(GX[..., 2], cmap='Reds', vmin=0, vmax=1)
    axes[0].set_title('GOES fire, last 24 h')
    for ax, (h, (_, _, p_f)) in zip(axes[1:], GOES_OUT.items()):
        im = ax.imshow(p_f, cmap='YlOrRd', vmin=0, vmax=P_MAX)
        ax.contour(GOES_BURNED, levels=[0.5], colors='k', linewidths=0.8)
        ax.set_title(f'+{h} h')
    for ax in axes:
        ax.axis('off')
    fig.colorbar(im, ax=axes, shrink=0.8, label='fire probability')
    plt.suptitle(f'Forecast issued {GOES_T:%Y-%m-%d %H}Z')
    plt.show()